# Comprehending ontology structure — the `StructureExplorer`

`ontology_modeler.StructureExplorer` is a toolkit of SPARQL-backed methods for
understanding what is in the triple store: the classes, object/data properties, SKOS
relations, the class hierarchy *within* and *between* ontologies, and how object
properties wire ontologies together.

Every method returns a **pandas DataFrame** whose rows are JSON-serialisable, so each is a
ready-made **agent function-tool**. `StructureExplorer` runs over the shared `FusekiClient`,
so it shares connection and auth with the rest of the component.

### Two design decisions, grounded in the loaded FIBO data

1. **Scope by ontology *IRI namespace*** (`STRSTARTS` on the entity IRI), because that is
   the identifier an agent has — not a file or graph path.
2. **Transitive hierarchy queries are unqualified** (union default graph) so
   `rdfs:subClassOf+` crosses ontology boundaries; wrapping a path in `GRAPH ?g` evaluates
   it per file and silently drops cross-ontology ancestry.

**Prerequisites:** stack running with FIBO loaded; run from `src/Ontology Modeler/notebooks/`.

In [1]:
import sys
from pathlib import Path
MODULE_DIR = Path.cwd().parent
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from ontology_modeler import OntologyModeler

struct = OntologyModeler().structure        # a StructureExplorer over the shared client
print("connected:", struct.client.settings.sparql_endpoint)
print("triples in store:", struct.client.count_triples())

connected: http://localhost:3030/ontology/sparql


triples in store: 133166


## 1. Inventory — what is in here?

Start with the ontologies, then enumerate classes and properties, optionally scoped to one
ontology. `list_ontologies()` gives you the `ontology` IRIs the other methods take as scope.

In [2]:
ontos = struct.list_ontologies()
print(f"{len(ontos)} ontologies loaded")
ontos.head(8)

299 ontologies loaded


,ontology,label,graph
0,https://spec.edmcouncil.org/fibo/ontology/ACTU...,ACTUS Challenge Examples Ontology,urn:graph:Ontology%20Repository/FIBO/fibo/ACTU...
1,https://spec.edmcouncil.org/fibo/ontology/ACTU...,ACTUS Contract Terms,urn:graph:Ontology%20Repository/FIBO/fibo/ACTU...
2,https://spec.edmcouncil.org/fibo/ontology/ACTU...,ACTUS Controlled Vocabulary,urn:graph:Ontology%20Repository/FIBO/fibo/ACTU...
3,https://spec.edmcouncil.org/fibo/ontology/ACTU...,ACTUS Contract Terms,urn:graph:Ontology%20Repository/FIBO/fibo/ACTU...
4,https://spec.edmcouncil.org/fibo/ontology/ACTU...,Metadata about the FIBO Algorithmic Contract T...,urn:graph:Ontology%20Repository/FIBO/fibo/ACTU...
5,https://spec.edmcouncil.org/fibo/ontology/Abou...,About FIBO Development,urn:graph:Ontology%20Repository/FIBO/fibo/Abou...
6,https://spec.edmcouncil.org/fibo/ontology/Abou...,About FIBO Production - including Reference Data,urn:graph:Ontology%20Repository/FIBO/fibo/Abou...
7,https://spec.edmcouncil.org/fibo/ontology/Abou...,About FIBO Production - T-Box Only,urn:graph:Ontology%20Repository/FIBO/fibo/Abou...


In [3]:
CAA = "https://spec.edmcouncil.org/fibo/ontology/FBC/ProductsAndServices/ClientsAndAccounts/"
print("all classes in store:", len(struct.classes()))
print("classes in Clients & Accounts:", len(struct.classes(CAA)))
struct.classes(CAA).head(8)

all classes in store: 3017
classes in Clients & Accounts: 47


,class,label,definition
0,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account,container for records associated with a busine...
1,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account as an asset,financial asset in the form of an account
2,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account holder,party that owns an account
3,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account identifier,identifier that denotes an account
4,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account ownership,holding of an account
5,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account provider,party that provides and services an account
6,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account statement,periodic summary of account activity for a giv...
7,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account-specific service agreement,"service-agreement that is account-specific, ap..."


In [4]:
print("object properties:", len(struct.object_properties()))
print("data properties:  ", len(struct.data_properties()))
struct.object_properties().head(6)

object properties: 948
data properties:   264


,property,label,domain,range
0,https://spec.edmcouncil.org/fibo/ontology/BE/C...,None,None,None
1,https://spec.edmcouncil.org/fibo/ontology/BE/C...,None,None,None
2,https://spec.edmcouncil.org/fibo/ontology/BE/C...,None,None,None
3,https://spec.edmcouncil.org/fibo/ontology/BE/O...,None,None,None
4,https://spec.edmcouncil.org/fibo/ontology/BE/O...,None,None,None
5,https://spec.edmcouncil.org/fibo/ontology/BE/O...,None,None,None


### SKOS relations

FIBO barely uses the SKOS *hierarchy* — its concept tree is `rdfs:subClassOf` (section 2),
and SKOS shows up mostly as annotations (`skos:definition`, `skos:changeNote`) plus a little
`skos:isNarrowerThan`. `skos_relations()` still works for vocabularies that use
broader/narrower/exactMatch, and `skos_relation_summary()` maps how much SKOS is in play.

In [5]:
print("SKOS semantic/mapping relations present:", len(struct.skos_relations()))
struct.skos_relation_summary()

SKOS semantic/mapping relations present: 64


,predicate,count
0,http://www.w3.org/2004/02/skos/core#definition,5989
1,http://www.w3.org/2004/02/skos/core#changeNote,1689
2,http://www.w3.org/2004/02/skos/core#note,1609
3,http://www.w3.org/2004/02/skos/core#example,177
4,http://www.w3.org/2004/02/skos/core#editorialNote,93
5,http://www.w3.org/2004/02/skos/core#isNarrower...,64
6,http://www.w3.org/2004/02/skos/core#scopeNote,40
7,http://www.w3.org/2004/02/skos/core#altLabel,13
8,http://www.w3.org/2004/02/skos/core#historyNote,4


## 2. Hierarchy analysis

Direct vs full ancestry/descendants, roots and leaves, and the **cross-ontology edges** that
are the most interesting structural signal. `direct=False` uses `rdfs:subClassOf+`,
unqualified, so ancestry crosses files.

In [6]:
DEPOSIT = "https://spec.edmcouncil.org/fibo/ontology/FBC/ProductsAndServices/ClientsAndAccounts/DepositAccount"
print("Deposit Account — direct parents:")
print(struct.superclasses(DEPOSIT, direct=True).to_string(index=False))
print("\nDeposit Account — full ancestry (crosses ontologies):")
struct.superclasses(DEPOSIT, direct=False)

Deposit Account — direct parents:
                                                                                                     superclass                         label
      https://spec.edmcouncil.org/fibo/ontology/FBC/FunctionalEntities/FinancialServicesEntities/BankingProduct               banking product
https://spec.edmcouncil.org/fibo/ontology/FBC/ProductsAndServices/ClientsAndAccounts/InvestmentOrDepositAccount investment or deposit account

Deposit Account — full ancestry (crosses ontologies):


,superclass,label
0,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account
1,https://spec.edmcouncil.org/fibo/ontology/FBC/...,banking product
2,https://spec.edmcouncil.org/fibo/ontology/FBC/...,financial product
3,https://spec.edmcouncil.org/fibo/ontology/FBC/...,investment or deposit account
4,https://spec.edmcouncil.org/fibo/ontology/FND/...,product


In [7]:
rl = struct.roots_and_leaves(CAA)
print(f"Clients & Accounts: {len(rl)} classes  "
      f"({(rl.is_root=='true').sum()} roots, {(rl.is_leaf=='true').sum()} leaves)")
rl.head(10)

Clients & Accounts: 47 classes  (1 roots, 28 leaves)


,class,label,is_root,is_leaf
0,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account,true,false
1,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account as an asset,false,true
2,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account holder,false,false
3,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account identifier,false,false
4,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account ownership,false,true
5,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account provider,false,false
6,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account statement,false,true
7,https://spec.edmcouncil.org/fibo/ontology/FBC/...,account-specific service agreement,false,false
8,https://spec.edmcouncil.org/fibo/ontology/FBC/...,accounting transaction,false,true
9,https://spec.edmcouncil.org/fibo/ontology/FBC/...,balance,false,true


In [8]:
edges = struct.cross_ontology_subclass_edges(min_edges=5)
print(f"{len(edges)} ontology pairs linked by >=5 cross-ontology subclass edges")
edges.head(10)

74 ontology pairs linked by >=5 cross-ontology subclass edges


,child_ontology,parent_ontology,edges
0,https://spec.edmcouncil.org/fibo/ontology/SEC/...,https://spec.edmcouncil.org/fibo/ontology/SEC/...,896
1,https://spec.edmcouncil.org/fibo/ontology/SEC/...,https://spec.edmcouncil.org/fibo/ontology/SEC/...,220
2,https://spec.edmcouncil.org/fibo/ontology/CAE/...,https://spec.edmcouncil.org/fibo/ontology/CAE/...,42
3,https://spec.edmcouncil.org/fibo/ontology/FBC/...,https://spec.edmcouncil.org/fibo/ontology/FBC/...,27
4,https://spec.edmcouncil.org/fibo/ontology/BP/S...,https://spec.edmcouncil.org/fibo/ontology/BP/S...,20
5,https://spec.edmcouncil.org/fibo/ontology/FBC/...,https://spec.edmcouncil.org/fibo/ontology/FBC/...,20
6,https://spec.edmcouncil.org/fibo/ontology/FBC/...,https://spec.edmcouncil.org/fibo/ontology/FND/...,18
7,https://spec.edmcouncil.org/fibo/ontology/BP/S...,https://spec.edmcouncil.org/fibo/ontology/BP/S...,17
8,https://spec.edmcouncil.org/fibo/ontology/FND/...,https://spec.edmcouncil.org/fibo/ontology/FND/...,16
9,https://spec.edmcouncil.org/fibo/ontology/SEC/...,https://spec.edmcouncil.org/fibo/ontology/SEC/...,16


## 3. Object-property analysis

Domain/range structure, and which object properties **bridge two ontologies** by relating a
class in one to a class in another.

In [9]:
print("object properties with named domain+range:", len(struct.property_domain_range()))
print("\nObject properties bridging ontologies:")
struct.cross_ontology_object_properties().head(10)

object properties with named domain+range: 560

Object properties bridging ontologies:


,domain_ontology,range_ontology,properties
0,https://spec.edmcouncil.org/fibo/ontology/SEC/...,https://spec.edmcouncil.org/fibo/ontology/SEC/...,14
1,https://spec.edmcouncil.org/fibo/ontology/SEC/...,https://spec.edmcouncil.org/fibo/ontology/SEC/...,11
2,https://spec.edmcouncil.org/fibo/ontology/SEC/...,https://spec.edmcouncil.org/fibo/ontology/FND/...,10
3,https://spec.edmcouncil.org/fibo/ontology/SEC/...,https://spec.edmcouncil.org/fibo/ontology/MD/C...,7
4,https://spec.edmcouncil.org/fibo/ontology/SEC/...,https://spec.edmcouncil.org/fibo/ontology/FND/...,6
5,https://spec.edmcouncil.org/fibo/ontology/SEC/...,https://spec.edmcouncil.org/fibo/ontology/FBC/...,6
6,https://spec.edmcouncil.org/fibo/ontology/SEC/...,https://spec.edmcouncil.org/fibo/ontology/SEC/...,6
7,https://spec.edmcouncil.org/fibo/ontology/BP/S...,https://spec.edmcouncil.org/fibo/ontology/SEC/...,5
8,https://spec.edmcouncil.org/fibo/ontology/SEC/...,https://spec.edmcouncil.org/fibo/ontology/MD/C...,5
9,https://spec.edmcouncil.org/fibo/ontology/IND/...,https://spec.edmcouncil.org/fibo/ontology/FND/...,4


## 4. Cross-ontology connectivity — the dependency picture

`dependency_matrix()` folds the cross-ontology subclass edges into a child×parent matrix.
A non-zero cell [A, B] means classes in A are subclasses of classes in B — i.e. A depends on
B. This is the signal `owl:imports` *intends* to declare, measured from the actual asserted
structure.

In [10]:
matrix = struct.dependency_matrix()
print(f"dependency matrix: {matrix.shape[0]} child x {matrix.shape[1]} parent ontologies")
busiest_children = matrix.astype(bool).sum(axis=1).sort_values(ascending=False).head(8).index
busiest_parents  = matrix.astype(bool).sum(axis=0).sort_values(ascending=False).head(8).index
matrix.loc[busiest_children, busiest_parents]

dependency matrix: 137 child x 108 parent ontologies


parent,FND/Agreements/Contracts,FBC/ProductsAndServices/FinancialProd...,FBC/FinancialInstruments/FinancialIns...,FND/DatesAndTimes/FinancialDates,FND/Accounting/CurrencyAmount,FND/DatesAndTimes/Occurrences,FND/Utilities/Analytics,SEC/Securities/SecuritiesIssuance
child,,,,,,,,
SEC/Debt/Bonds,1,2,2,0,1,0,1,2
SEC/Equities/EquityInstruments,0,2,4,1,1,1,0,4
FBC/DebtAndEquities/Debt,8,1,2,2,4,2,1,0
SEC/Funds/CollectiveInvestmentVehicles,1,0,0,0,1,0,1,1
SEC/Debt/CollateralizedDebtObligations,0,0,2,0,0,0,0,1
DER/DerivativesContracts/DerivativesB...,3,2,6,0,0,0,2,1
FBC/ProductsAndServices/ClientsAndAcc...,0,2,1,2,2,1,0,0
SEC/Funds/Funds,1,1,3,0,0,0,0,0


## Summary — wrapping these as agent tools

**Inventory:** `list_ontologies`, `classes`, `object_properties`, `data_properties`,
`skos_relations` (+ `skos_relation_summary`).
**Hierarchy:** `superclasses`, `subclasses`, `roots_and_leaves`, `cross_ontology_subclass_edges`.
**Object properties:** `property_domain_range`, `cross_ontology_object_properties`.
**Connectivity:** `dependency_matrix`.

Every method returns a DataFrame, so exposing one as a tool is mechanical:

```python
from ontology_modeler import OntologyModeler
struct = OntologyModeler().structure

def tool_list_classes(ontology: str | None = None) -> list[dict]:
    "List classes in the store, optionally scoped to one ontology IRI."
    return struct.classes(ontology).fillna("").to_dict("records")
```

Two invariants the agent layer must keep, both learned from the loaded FIBO data: **scope by
ontology IRI namespace**, and **keep transitive hierarchy queries unqualified** so ancestry
crosses ontology boundaries.